# Solutions · 02 · Frequency response

Worked solutions to the exercises of [notebook 02](../notebooks/02_frequency_response.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, frequency as fr, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Add a pure delay of 0.1 s to L (`fr.freqresp(L, w, delay=0.1)` and `fr.margins(L, delay=0.1)`): how do the margins change, and does the gain margin or the phase margin react first? Compare with the delay margin computed before.

In [2]:
L = sy.zpk([-10.0], [0.0, -1.0, -50.0], 100.0)
m0 = fr.margins(L)
print(f"no delay: gain margin {m0['gain_margin']}, phase margin {m0['phase_margin']:.1f} deg, delay margin {m0['delay_margin'] * 1000:.0f} ms")
for Td in (0.05, 0.1, 0.12):
    m = fr.margins(L, delay=Td)
    print(f"delay {Td * 1000:4.0f} ms: gain margin {m['gain_margin']:.2f}, phase margin {m['phase_margin']:.1f} deg")

no delay: gain margin inf, phase margin 31.7 deg, delay margin 120 ms
delay   50 ms: gain margin 6.64, phase margin 18.5 deg


delay  100 ms: gain margin 1.49, phase margin 5.2 deg
delay  120 ms: gain margin 0.99, phase margin -0.1 deg


The delay subtracts ωT of phase: the phase margin falls first (by ω_gc T, 27° for 100 ms), while a finite gain
margin appears only once the phase reaches −180° somewhere. At the delay margin (120 ms) the phase margin
reaches zero - the delay margin computed without the delay predicted exactly where the loop would fail.

## Exercise 2

Sketch by hand the Bode asymptotes of the DC motor's voltage-to-speed transfer function and check them against `fr.bode`. Where is its bandwidth, and what sets it?

In [3]:
G = models.dc_motor()["tf_speed"]
print("G =", G, "-> dc gain", round(G.dcgain(), 4), "= -20 dB and corners at the poles", np.round(-G.poles().real, 2), "rad/s")
w = np.logspace(-1, 3, 400)
_, mag, ph = fr.bode(G, w)
asym = 20 * np.log10(G.dcgain()) - 20 * np.log10(np.maximum(w / 2.0, 1)) - 20 * np.log10(np.maximum(w / 10.0, 1))
print(f"asymptotes: flat at {20 * np.log10(G.dcgain()):.1f} dB, then -20 dB/dec from 2 rad/s and -40 dB/dec from 10 rad/s; max deviation {np.max(np.abs(mag - asym)):.2f} dB")
print(f"bandwidth (-3 dB) {fr.bandwidth(G):.2f} rad/s: set by the mechanical pole at 2 rad/s (the slow one)")

G = TransferFunction(2 / s^2 + 12s + 20.02) -> dc gain 0.0999 = -20 dB and corners at the poles [10.  2.] rad/s
asymptotes: flat at -20.0 dB, then -20 dB/dec from 2 rad/s and -40 dB/dec from 10 rad/s; max deviation 3.15 dB
bandwidth (-3 dB) 1.92 rad/s: set by the mechanical pole at 2 rad/s (the slow one)


Two real poles give two corners; the bandwidth is at the first of them, the mechanical time constant - the
motor's speed cannot follow voltage variations faster than about 2 rad/s, whatever the electrical pole does.

## Exercise 3

**Implement it yourself:** compute the Nyquist curve of L from the state-space model with C (jwI - A)^-1 B + D (one linear solve per frequency) and compare with the transfer-function evaluation; which is better conditioned for a 10th-order system?

In [4]:
n = 10
poles = -np.arange(1, n + 1, dtype=float)
G = sy.zpk([], poles, 1.0)
S = G.to_ss()
w = np.logspace(-1, 2, 200)
H_tf = fr.freqresp(G, w)
H_ss = fr.freqresp(S, w)
exact = np.prod([1 / (1j * w - p) for p in poles], axis=0)
print(f"10th order: transfer-function evaluation error {np.max(np.abs(H_tf - exact) / np.abs(exact)):.1e}, state-space (companion) error {np.max(np.abs(H_ss - exact) / np.abs(exact)):.1e}")
bal, _ = sy.balanced_realisation(S)
print(f"balanced state-space error {np.max(np.abs(fr.freqresp(bal, w) - exact) / np.abs(exact)):.1e}")

10th order: transfer-function evaluation error 9.8e-16, state-space (companion) error 2.4e-15


balanced state-space error 8.4e+00


For a 10th-order system both the polynomial evaluation (Horner's scheme is stable for an imaginary argument)
and the companion-form linear solve are accurate to rounding - and the balanced realisation, computed from
the companion form through two Gramians, is wrong by a factor of 8: its transformation holds inverse fourth
roots of Hankel singular values spanning many orders of magnitude and injects the error (notebook 16 meets
this again). The representation one *computes into* matters as much as the one one evaluates in.